# YouTube 댓글 데이터 토큰화 및 형태소 분석 파이프라인

이 노트북은 정제된 YouTube 댓글 데이터를 로드하여 `Kiwi` 형태소 분석기를 사용해 동사와 형용사를 추출하고, 연구 목적에 맞게 토큰을 가공하는 과정을 담고 있습니다.

## 1. 환경 설정 및 라이브러리 설치

In [1]:
!pip install kiwipiepy pandas tqdm

In [2]:
import pandas as pd
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords
from tqdm import tqdm
import os

# 결과 확인을 위한 설정
pd.set_option('display.max_colwidth', None)

## 2. 데이터 로드

- 앞선 단계에서 전처리가 완료된 데이터를 불러옵니다.

In [3]:
input_file = os.path.join("..", "data", "processed", "토크나이징_전_전처리.csv")
df = pd.read_csv(input_file, encoding='utf-8-sig')
df['cleaned_text'] = df['cleaned_text'].fillna("")
print(f"로드된 데이터 수: {len(df)}")

로드된 데이터 수: 74356


C:\Users\user\AppData\Local\Temp\ipykernel_18764\2047094123.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("preprocessed_comments.csv", encoding='utf-8-sig')


## 3. Kiwi 형태소 분석 및 품사 추출

- **추출 대상**: 
    - `VV`: 동사 (원형 추출)
    - `VA`: 형용사 (원형 추출)
    - `NNG`, `NNP`: 일반/고유 명사 (선택 사항)
- **연구용 고도화 설정**:
    - `normalize_coda=True`: '고마워요' -> '고맙다' 등 구어체 정규화
    - `Stopwords`: 불용어 필터링
    - 한 글자 단어 제거 및 불완전 자모 제거 규칙 적용

In [ ]:
kiwi = Kiwi()

# 불용어 설정
stopwords = Stopwords()
custom_stopwords = ['하다', '있다', '되다', '않다', '같다', '싶다', '이렇다', '저렇다', '어떻다', '나오다', '가지다', '보다', '해주다', '그렇다', '알다', '모르다', '생각하다']
for word in custom_stopwords:
    stopwords.add((word, 'VV'))
    stopwords.add((word, 'VA'))

def tokenize(text):
    if not text:
        return [], []
    result = kiwi.analyze(text)
    tokens_raw, _ = result[0]
    forms, tags = [], []
    for token in tokens_raw:
        form = token.form
        tag = str(token.tag)
        if form and ord(form[0]) >= 0x11A8 and ord(form[0]) <= 0x11FF:
            continue
        if len(form) >= 2:
            forms.append(form)
            tags.append(tag)
    return forms, tags

def extract_pos(text, target_tags=['VV', 'VA']):  #
    if not text: return ""
    results = kiwi.tokenize(text, stopwords=stopwords, normalize_coda=True)
    tokens = [token.form for token in results if token.tag in target_tags and len(token.form) > 1]
    return " ".join(tokens)




In [5]:
print("형태소 분석 및 토큰 결합 중...")
results = df['cleaned_text'].apply(tokenize)
df['tokens'] = results.apply(lambda x: x[0])
df['pos_tags'] = results.apply(lambda x: x[1])

df['tokens_str'] = df['tokens'].apply(lambda t: ' '.join(t))
df['tokens_pos_str'] = df.apply(
    lambda row: ' '.join(f"{f}/{p}" for f, p in zip(row['tokens'], row['pos_tags'])),
    axis=1
)

tqdm.pandas()
print("연구용 동사/형용사(VV/VA) 추출 중...")
df['extracted_tokens'] = df['cleaned_text'].progress_apply(lambda x: extract_pos(x, target_tags=['VV', 'VA']))

# 필요 시 명사도 추출할 수 있음
# df['extracted_tokens'] = df['cleaned_text'].progress_apply(lambda x: extract_pos(x, target_tags=['VV', 'VA', 'NNG', 'NNP']))

print("형태소 분석 완료")


형태소 분석 및 토큰 결합 중...
연구용 동사/형용사(VV/VA) 추출 중...


100%|██████████| 74356/74356 [01:51<00:00, 667.19it/s] 

형태소 분석 완료


## 4. 데이터 검증 및 분석용 필터링

- 추출된 토큰이 없는 빈 행 제거
- 빈도 분석을 통한 불용어 추가 발굴 (연구의 타당성 검토)

In [6]:
df_final = df[df['extracted_tokens'] != ""].copy()
print(f"Final data count for analysis: {len(df_final)}")

from collections import Counter
all_tokens = " ".join(df_final['extracted_tokens']).split()
token_counts = Counter(all_tokens)
print("\nTop 50 Tokens (Check for additional stopwords):")
for i, (word, count) in enumerate(token_counts.most_common(50)):
    print(f"{i+1}. {word}: {count}")

Final data count for analysis: 46048

Top 50 Tokens (Check for additional stopwords):
1. 힘들: 15826
2. 모르: 6476
3. 나오: 5564
4. 느끼: 3710
5. 괜찮: 3294
6. 아프: 2937
7. 힘내: 2728
8. 그러: 2718
9. 버티: 2462
10. 보이: 2242
11. 심하: 2204
12. 슬프: 2017
13. 생기: 1891
14. 살아가: 1873
15. 만나: 1832
16. 나가: 1722
17. 걸리: 1695
18. 바라: 1672
19. 다니: 1548
20. 가지: 1504
21. 어리: 1499
22. 일어나: 1418
23. 지치: 1411
24. 보내: 1370
25. 태어나: 1367
26. 나아지: 1346
27. 이기: 1330
28. 만들: 1286
29. 빠지: 1282
30. 좋아하: 1268
31. 지나: 1208
32. 편하: 1188
33. 사라지: 1148
34. 이러: 1138
35. 드리: 1138
36. 지내: 1058
37. 미치: 1035
38. 떨어지: 911
39. 다르: 908
40. 나쁘: 876
41. 견디: 839
42. 돌아가: 834
43. 지나가: 829
44. 떠나: 738
45. 벗어나: 706
46. 당하: 672
47. 귀찮: 670
48. 들어오: 669
49. 어쩌: 666
50. 똑같: 666


## 5. 결과 저장

- LDA 분석 및 후속 연구를 위해 전처리 및 토큰화가 완료된 파일들을 각각 분리하여 저장합니다.

In [7]:
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)

out = df[['cid', 'tokens_str', 'tokens_pos_str']].copy()
out = out[out['tokens_str'].str.strip() != '']
out_path = os.path.join(output_dir, "tokens_only.csv")
out.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"저장 완료: {out_path}")
print(f"유효 댓글 수: {len(out)}")

output_file = os.path.join(output_dir, 'Cleaned_YoutubeComments_Only.csv')
df_final[['extracted_tokens']].to_csv(output_file, index=False, encoding='utf-8-sig', header=['cleaned_text'])
print(f"저장 완료: {output_file}")

저장 완료: tokens_only.csv
유효 댓글 수: 69049
저장 완료: Cleaned_YoutubeComments_Only.csv


In [8]:
print("\n--- 명사 포함(VV/VA/NNG/NNP) 추가 추출 시작 ---")

# 1. 타겟 태그에 명사(NNG: 일반명사, NNP: 고유명사)를 추가하여 progress_apply 실행
df['extracted_tokens_with_nouns'] = df['cleaned_text'].progress_apply(
    lambda x: extract_pos(x, target_tags=['VV', 'VA', 'NNG', 'NNP'])
)

# 2. 토큰 추출 결과가 빈 문자열인 행 제거 (명사 포함 버전용)
df_final_nouns = df[df['extracted_tokens_with_nouns'] != ""].copy()
print(f"명사 포함 분석 데이터 수: {len(df_final_nouns)}")

# 3. 명사 포함 버전 Top 50 빈도 분석 (검증용)
all_tokens_nouns = " ".join(df_final_nouns['extracted_tokens_with_nouns']).split()
token_counts_nouns = Counter(all_tokens_nouns)
print("\nTop 50 Tokens (명사 포함 버전 - 불용어 확인용):")
for i, (word, count) in enumerate(token_counts_nouns.most_common(50)):
    print(f"{i+1}. {word}: {count}")

# 4. 다른 이름으로 결과 파일 저장
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_file_nouns = os.path.join(output_dir, 'Cleaned_YoutubeComments_WithNouns.csv')  # 파일명 구분
df_final_nouns[['extracted_tokens_with_nouns']].to_csv(
    output_file_nouns, 
    index=False, 
    encoding='utf-8-sig', 
    header=['cleaned_text']
)

print(f"🎉 저장 완료 (명사 포함): {output_file_nouns}")
print("--- 모든 추가 작업이 완료되었습니다! ---")


--- 명사 포함(VV/VA/NNG/NNP) 추가 추출 시작 ---


100%|██████████| 74356/74356 [01:52<00:00, 661.39it/s] 


명사 포함 분석 데이터 수: 64685

Top 50 Tokens (명사 포함 버전 - 불용어 확인용):
1. 생각: 16360
2. 힘들: 15826
3. 우울증: 14491
4. 마음: 7877
5. 우울: 7052
6. 모르: 6476
7. 행복: 6360
8. 나오: 5564
9. 친구: 4696
10. 감사: 4632
11. 자신: 4222
12. 시간: 4009
13. 느끼: 3710
14. 지금: 3471
15. 괜찮: 3294
16. 자살: 3244
17. 병원: 3111
18. 인생: 3071
19. 감정: 2960
20. 아프: 2937
21. 영상: 2928
22. 세상: 2912
23. 엄마: 2906
24. 사랑: 2903
25. 치료: 2898
26. 가족: 2821
27. 부모: 2783
28. 상담: 2738
29. 힘내: 2728
30. 그러: 2718
31. 하루: 2640
32. 혼자: 2621
33. 눈물: 2604
34. 정도: 2588
35. 위로: 2564
36. 버티: 2462
37. 도움: 2436
38. 이상: 2311
39. 보이: 2245
40. 댓글: 2223
41. 심하: 2204
42. 고통: 2119
43. 공부: 2098
44. 요즘: 2068
45. 노력: 2052
46. 기분: 2045
47. 오늘: 2030
48. 슬프: 2017
49. 말씀: 1994
50. 정신: 1962
🎉 저장 완료 (명사 포함): Cleaned_YoutubeComments_WithNouns.csv
--- 모든 추가 작업이 완료되었습니다! ---
